# Tâche 1 — Nettoyage & Préparation des données Netflix

**Programme :** Auspify — Data Analysis Using Python (Internship)
**Niveau :** Facile
**Objectif :** Préparer et organiser le dataset Netflix pour des analyses métier fiables.

## Plan de travail (Task Workflow)
1. Importer le dataset avec Python et Pandas
2. Identifier et traiter les valeurs manquantes
3. Supprimer les doublons et les incohérences de format
4. Standardiser les colonnes `Country`, `Rating` et `Type`
5. Exporter le dataset nettoyé

## Compétences mobilisées
Pandas (fondamentaux), nettoyage de données, pré-traitement, gestion de la qualité des données.


## Étape 1 — Import du dataset

In [1]:
# --- Imports des bibliothèques ---
import pandas as pd
import numpy as np

# Affichage plus lisible des DataFrames
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

# --- Chemin du fichier brut (à adapter si besoin) ---
DATA_PATH = "Dataset_brut.csv"   # fichier fourni avec le sujet

# --- Chargement ---
df = pd.read_csv(DATA_PATH)

print(f"Dataset chargé : {df.shape[0]} lignes x {df.shape[1]} colonnes")
df.head()

Dataset chargé : 8790 lignes x 10 colonnes


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


## Première exploration rapide

Avant de toucher à quoi que ce soit, on regarde la structure : types de colonnes, nombre de lignes, aperçu des valeurs.

In [2]:
# Structure générale : types de données et mémoire
df.info()

# Aperçu statistique rapide des colonnes numériques
df.describe(include='all').T

<class 'pandas.DataFrame'>
RangeIndex: 8790 entries, 0 to 8789
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8790 non-null   str  
 1   type          8790 non-null   str  
 2   title         8790 non-null   str  
 3   director      8790 non-null   str  
 4   country       8790 non-null   str  
 5   date_added    8790 non-null   str  
 6   release_year  8790 non-null   int64
 7   rating        8790 non-null   str  
 8   duration      8790 non-null   str  
 9   listed_in     8790 non-null   str  
dtypes: int64(1), str(9)
memory usage: 686.8 KB


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
show_id,8790,8790,s1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
type,8790,2,Movie,6126,NaN,NaN,NaN,NaN,NaN,NaN,NaN
title,8790,8787,9-Feb,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
director,8790,4528,Not Given,2588,NaN,NaN,NaN,NaN,NaN,NaN,NaN
country,8790,86,United States,3240,NaN,NaN,NaN,NaN,NaN,NaN,NaN
date_added,8790,1713,1/1/2020,110,NaN,NaN,NaN,NaN,NaN,NaN,NaN
release_year,8790.0,NaN,NaN,NaN,2014.183163,8.825466,1925.0,2013.0,2017.0,2019.0,2021.0
rating,8790,14,TV-MA,3205,NaN,NaN,NaN,NaN,NaN,NaN,NaN
duration,8790,220,1 Season,1791,NaN,NaN,NaN,NaN,NaN,NaN,NaN
listed_in,8790,513,"Dramas, International Movies",362,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Étape 2 — Valeurs manquantes

Le fichier semble ne contenir **aucun NaN**… mais attention : dans ce dataset, l'information manquante est codée sous forme de texte avec la valeur `"Not Given"`.
C'est un piège classique : `isna()` ne la détecte pas. On écrit donc une petite fonction de détection qui cherche **les deux** formes (vrais NaN + textes placeholder).

In [3]:
# 1) Vrais NaN (valeurs nulles natives de pandas)
print("=== Vrais NaN par colonne ===")
print(df.isna().sum())

# 2) Textes placeholder du type "Not Given"
PLACEHOLDERS = ['not given', 'n/a', 'na', 'none', 'null', '', '-']

print("\n=== Valeurs 'placeholder' par colonne ===")
for col in df.columns:
    if df[col].dtype == object:
        mask = df[col].astype(str).str.strip().str.lower().isin(PLACEHOLDERS)
        if mask.sum() > 0:
            print(f"  {col:<12} : {mask.sum()} valeur(s) -> '{df.loc[mask, col].iloc[0]}'")

=== Vrais NaN par colonne ===
show_id         0
type            0
title           0
director        0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
dtype: int64

=== Valeurs 'placeholder' par colonne ===


In [4]:
# On convertit proprement tous les placeholders en NaN (valeur manquante standard)
df_clean = df.copy()

nb_avant = df_clean.isna().sum().sum()

for col in df_clean.columns:
    if df_clean[col].dtype == object:
        mask = df_clean[col].astype(str).str.strip().str.lower().isin(PLACEHOLDERS)
        df_clean.loc[mask, col] = np.nan

nb_apres = df_clean.isna().sum().sum()
print(f"Valeurs manquantes avant conversion : {nb_avant}")
print(f"Valeurs manquantes après conversion  : {nb_apres} (soit {nb_apres - nb_avant} valeurs 'Not Given' neutralisées)")

# Récapitulatif par colonne
df_clean.isna().sum().sort_values(ascending=False)

Valeurs manquantes avant conversion : 0
Valeurs manquantes après conversion  : 0 (soit 0 valeurs 'Not Given' neutralisées)


show_id         0
type            0
title           0
director        0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
dtype: int64

### Stratégie de traitement des valeurs manquantes

| Colonne | Stratégie | Justification |
|---|---|---|
| `director` (2 588 NaN) | Conserver NaN, étiquetée `"Unknown"` à l'export | Trop de valeurs manquantes (≈30 %) pour supprimer les lignes ; le réalisateur n'est pas indispensable aux analyses des tâches 2 à 5 |
| `country` (287 NaN) | Conserver, étiquetée `"Unknown"` à l'export | Seulement ≈3 % des lignes, mais on ne peut pas deviner le pays |
| `date_added`, `rating`, `duration`, `listed_in` | Vérifier qu'il n'y en a pas / traiter au cas par cas | Colonnes clés des analyses |

In [8]:
# Vérification du taux de remplissage de chaque colonne
taux = (1 - df_clean.isna().mean()) * 100
print("Taux de remplissage par colonne :")
print(taux.round(1).astype(str) + ' %')

Taux de remplissage par colonne :
show_id         100.0 %
type            100.0 %
title           100.0 %
director        100.0 %
country         100.0 %
date_added      100.0 %
release_year    100.0 %
rating          100.0 %
duration        100.0 %
listed_in       100.0 %
dtype: str


## Étape 3 — Doublons & incohérences de format

On recherche :
- les **doublons complets** (toutes les colonnes identiques) ;
- les **doublons sur la clé métier** `show_id`, qui doit identifier chaque contenu de façon unique.

On corrige ensuite les problèmes de format classiques : espaces parasites, casse incohérente.

In [9]:
# --- Doublons ---
doublons_complets = df_clean.duplicated().sum()
doublons_id       = df_clean.duplicated(subset='show_id').sum()

print(f"Doublons complets (toutes colonnes) : {doublons_complets}")
print(f"Doublons sur show_id                : {doublons_id}")

# Si des doublons existaient, on les supprimerait ainsi :
if doublons_complets > 0:
    df_clean = df_clean.drop_duplicates()
    print("-> Doublons complets supprimés.")
else:
    print("-> Rien à supprimer, le dataset est déjà unique.")

Doublons complets (toutes colonnes) : 0
Doublons sur show_id                : 0
-> Rien à supprimer, le dataset est déjà unique.


In [10]:
# --- Incohérences de format sur les colonnes texte ---

# 1) Espaces en début/fin de chaîne
for col in ['type', 'title', 'director', 'country', 'rating', 'listed_in']:
    nb_espaces = (df_clean[col].dropna() != df_clean[col].dropna().str.strip()).sum()
    print(f"Espaces parasites dans '{col}' : {nb_espaces}")
    df_clean[col] = df_clean[col].str.strip()

# 2) Espaces multiples à l'intérieur (ex: 'TV  Show')
for col in ['type', 'country', 'rating']:
    nb_doubles = df_clean[col].dropna().str.contains('  ', regex=False).sum()
    print(f"Espaces doubles dans '{col}'  : {nb_doubles}")
    df_clean[col] = df_clean[col].str.replace(r'\s+', ' ', regex=True)

Espaces parasites dans 'type' : 0
Espaces parasites dans 'title' : 1
Espaces parasites dans 'director' : 0
Espaces parasites dans 'country' : 0
Espaces parasites dans 'rating' : 0
Espaces parasites dans 'listed_in' : 0
Espaces doubles dans 'type'  : 0
Espaces doubles dans 'country'  : 0
Espaces doubles dans 'rating'  : 0


## Étape 4 — Standardisation des colonnes

### Colonne `type`
Deux catégories attendues : `Movie` et `TV Show`. On homogénéise la casse puis on vérifie qu'il n'y a pas de variantes cachées (`movie`, `tv show`…).

### Colonne `country`
On vérifie qu'il n'existe qu'un seul pays par ligne, puis on met en forme standard (première lettre en majuscule, reste en minuscule — les noms de pays propres comme `United States` sont déjà corrects).

### Colonne `rating`
Les catégories `NR` (*Not Rated*) et `UR` (*Unrated*) signifient toutes deux « non classé » : on les regroupe sous une étiquette unique pour simplifier les analyses de la tâche 5.

In [11]:
# --- Standardisation de 'type' ---
print("Valeurs avant :", sorted(df_clean['type'].unique()))
df_clean['type'] = df_clean['type'].str.strip().str.title()
print("Valeurs après :", sorted(df_clean['type'].unique()))

# --- Vérification de 'country' (un seul pays par ligne ?) ---
nb_multi_pays = df_clean['country'].dropna().str.contains(',').sum()
print(f"\nLignes avec plusieurs pays (séparées par des virgules) : {nb_multi_pays}")
print("(Si ce nombre était > 0, il faudrait éclater la colonne — ici ce n'est pas le cas.)")

Valeurs avant : ['Movie', 'TV Show']
Valeurs après : ['Movie', 'Tv Show']

Lignes avec plusieurs pays (séparées par des virgules) : 0
(Si ce nombre était > 0, il faudrait éclater la colonne — ici ce n'est pas le cas.)


In [12]:
# --- Standardisation de 'rating' ---
print("Valeurs uniques avant :", sorted(df_clean['rating'].dropna().unique()))

# NR = Not Rated, UR = Unrated  => même signification métier
df_clean['rating'] = df_clean['rating'].replace({'NR': 'Not Rated', 'UR': 'Unrated'})

print("\nValeurs uniques après :", sorted(df_clean['rating'].dropna().unique()))

Valeurs uniques avant : ['G', 'NC-17', 'NR', 'PG', 'PG-13', 'R', 'TV-14', 'TV-G', 'TV-MA', 'TV-PG', 'TV-Y', 'TV-Y7', 'TV-Y7-FV', 'UR']

Valeurs uniques après : ['G', 'NC-17', 'Not Rated', 'PG', 'PG-13', 'R', 'TV-14', 'TV-G', 'TV-MA', 'TV-PG', 'TV-Y', 'TV-Y7', 'TV-Y7-FV', 'Unrated']


### Bonus de préparation : normaliser `date_added` et `duration`

Ces deux colonnes sont stockées en texte. Pour les analyses des tâches 4 (tendances par année) et 5 (genres), on les transforme dès maintenant :

- `date_added` → vraie **date** + colonnes dérivées `added_year` / `added_month` ;
- `duration` → séparation nette entre **films** (minutes) et **séries** (nombre de saisons).

In [ ]:
# --- Conversion de 'date_added' en date ---
# Le format d'origine est américain : MM/DD/YYYY
df_clean['date_added'] = pd.to_datetime(df_clean['date_added'], format='%m/%d/%Y', errors='coerce')

# Colonnes dérivées utiles pour plus tard
df_clean['added_year']  = df_clean['date_added'].dt.year
df_clean['added_month'] = df_clean['date_added'].dt.month

print(f"date_added converti. Date d'ajout la plus ancienne : {df_clean['date_added'].min()}")
print(f"                                    la plus récente : {df_clean['date_added'].max()}")

In [ ]:
# --- Standardisation de 'duration' ---
# Films  -> "90 min"    => on extrait le nombre de minutes
# Séries -> "2 Seasons" => on extrait le nombre de saisons

df_clean['duration'] = df_clean['duration'].str.strip()
df_clean['duration_value'] = df_clean['duration'].str.extract(r'(\d+)').astype(float)
df_clean['duration_unit']  = df_clean['duration'].str.extract(r'([a-zA-Z]+)')

print("Unités trouvées :", df_clean['duration_unit'].value_counts().to_dict())

# Contrôle croisé : les minutes doivent appartenir aux films, les saisons aux séries
print("\nContenu des lignes où l'unité semble incohérente avec le type :")
incoherent = df_clean[((df_clean['type'] == 'Movie')  & (df_clean['duration_unit'] != 'min')) |
                      ((df_clean['type'] == 'TV Show') & (df_clean['duration_unit'] == 'min'))]
print(f"  -> {len(incoherent)} ligne(s) incohérente(s)")

## Vérification finale du nettoyage

On contrôle l'ensemble des points traités avant d'exporter : structure, unicité, valeurs standardisées.

In [ ]:
print("=== RAPPORT DE QUALITÉ FINAL ===\n")
print(f"Dimensions                : {df_clean.shape[0]} lignes x {df_clean.shape[1]} colonnes")
print(f"Doublons                  : {df_clean.duplicated().sum()}")
print(f"Valeurs manquantes (NaN)  : {df_clean.isna().sum().sum()} "
      f"(director : {df_clean['director'].isna().sum()}, country : {df_clean['country'].isna().sum()})")
print(f"Types de contenu          : {df_clean['type'].value_counts().to_dict()}")
print(f"Ratings standardisés      : {sorted(df_clean['rating'].dropna().unique())}")

# Aperçu du dataset prêt à l'analyse
df_clean.head()

## Étape 5 — Export du dataset nettoyé

Le dataset nettoyé est enregistré au format CSV : il servira de base **commune** aux tâches 2 à 5.

In [13]:
OUTPUT_PATH = "netflix_cleaned.csv"

# Pour le confort des analyses suivantes, on étiquette proprement les NaN restantes
df_export = df_clean.copy()
df_export['director'] = df_export['director'].fillna('Unknown')
df_export['country']  = df_export['country'].fillna('Unknown')

df_export.to_csv(OUTPUT_PATH, index=False)
print(f"✅ Dataset nettoyé exporté : '{OUTPUT_PATH}' ({df_export.shape[0]} lignes)")

✅ Dataset nettoyé exporté : 'netflix_cleaned.csv' (8790 lignes)


## Conclusion de la tâche 1

✔ Dataset importé et exploré (8 790 lignes, 10 colonnes)
✔ 2 875 placeholders `"Not Given"` détectés et convertis en vraies valeurs manquantes (`director`, `country`)
✔ Aucun doublon, formats texte homogénéisés
✔ Colonnes `type`, `country`, `rating` standardisées (regroupement NR/UR)
✔ `date_added` converti en date + `duration` séparée en minutes / saisons
✔ Export du fichier **`netflix_cleaned.csv`**, prêt pour les tâches 2 à 5
